### Imports

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import pickle
from IPython.display import display, clear_output
import warnings
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Image, display

### Configs

In [ ]:
generation_cap = 5.  # Placeholder
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('future.no_silent_downcasting', True)
warnings.simplefilter("ignore", DeprecationWarning)
pd.options.mode.chained_assignment = None  # default='warn'

### Constants

In [ ]:
# NATIONAL POKEDEX
GEN_RANGES = {
    1: range(1, 152),
    2: range(152, 252),
    3: range(251, 387),
    4: range(386, 494),
    5: range(493, 650),
    6: range(649, 722),
    7: range(721, 810),
    8: range(809, 906),
    9: range(905, 1026)
}

# Add new gens as necessary
RENAME_ALTERNATE_FORMS = {
    'GIRATINA ALTERED FORME': 'GIRATINA',
    'GIRATINA ORIGIN FORME': 'GIRATINA DISTORTION FORME',
    'DARMANITAN STANDARD MODE': 'DARMANITAN',
    'BASCULIN RED-STRIPED FORM': 'BASCULIN',
    'TORNADUS INCARNATE FORME': 'TORNADUS',
    'THUNDURUS INCARNATE FORME': 'THUNDURUS',
    'LANDORUS INCARNATE FORME': 'LANDORUS',
    'KYUREM WHITE KYUREM': 'WHITE KYUREM',
    'KYUREM BLACK KYUREM': 'BLACK KYUREM',
    'MELOETTA ARIA FORME': 'MELOETTA',
    'KELDEO ORDINARY FORM': 'KELDEO'
}

ALTERNATE_FORMS = {
    'MEGA': 6, 
    'GALARIAN': 8, 
    'HISUIAN': 8, 
    'ALOLAN': 7, 
    'PALDEAN': 9, 
    'PRIMAL': 6, 
    'PARTNER': 'remove', 
    'BREED': 9, 
    'ORIGIN': 8
    }

REDUNDANT_POKEMON = ['BASCULIN BLUE-STRIPED FORM', 'BASCULIN WHITE-STRIPED FORM', 'KELDEO RESOLUTE FORM']

# Add new gens 
REGIONAL_POKEDEX_URLS = {
    'Johto': 'https://bulbapedia.bulbagarden.net/wiki/List_of_Pok%C3%A9mon_by_Johto_Pok%C3%A9dex_number',  # Based on gen 4 remakes
    'Hoenn': 'https://bulbapedia.bulbagarden.net/wiki/List_of_Pok%C3%A9mon_by_Hoenn_Pok%C3%A9dex_number_in_Generation_III',
    'Sinnoh': 'https://bulbapedia.bulbagarden.net/wiki/List_of_Pok%C3%A9mon_by_Sinnoh_Pok%C3%A9dex_number',
    'Unova (Black/White)':' https://bulbapedia.bulbagarden.net/wiki/List_of_Pok%C3%A9mon_by_Unova_Pok%C3%A9dex_number_in_Pok%C3%A9mon_Black_and_White',
    'Unova (Black2/White2)': 'https://bulbapedia.bulbagarden.net/wiki/List_of_Pok%C3%A9mon_by_Unova_Pok%C3%A9dex_number_in_Pok%C3%A9mon_Black_2_and_White_2',
}

# Add new gens
# Used to filter out unnecessary row data from each regional pokedex
REGIONAL_INDEX_CAP = {
    'Johto': 288,  # Based on gen 4 remake
    'Hoenn': 210,
    'Sinnoh': 253,
    'Unova (Black/White)': 172,
    'Unova (Black2/White2)': 329
}

# Correct namings in regional pokedex dataframes prior to national dex join
REGIONAL_POKEDEX_RENAMES = {
    'UNOWNONE FORM': 'UNOWN',
    'BURMYPLANT CLOAK': 'BURMY PLANT CLOAK',
    'WORMADAMPLANT CLOAK': 'WORMADAM PLANT CLOAK',
    'SHELLOSWEST SEA': 'SHELLOS',
    'CHERRIMOVERCAST FORM': 'CHERRIM',
    'GASTRODONWEST SEA': 'GASTRODON',
    'ROTOMROTOM': 'ROTOM',
    'GIRATINAALTERED FORME': 'GIRATINA',
    'UNFEZANTMALE': 'UNFEZANT',
    'BASCULINRED-STRIPED FORM': 'BASCULIN',
    'DARMANITANSTANDARD MODE': 'DARMANITAN',
    'DEERLINGSPRING FORM': 'DEERLING',
    'SAWSBUCKSPRING FORM': 'SAWSBUCK',
    'MELOETTAARIA FORME': 'MELOETTA',
    'CASTFORMNORMAL': 'CASTFORM',
    'FRILLISHMALE': 'FRILLISH',
    'JELLICENTMALE': 'JELLICENT',
    'TORNADUSINCARNATE FORME': 'TORNADUS',
    'THUNDURUSINCARNATE FORME': 'THUNDURUS',
    'LANDORUSINCARNATE FORME': 'LANDORUS',
    'KYUREMKYUREM': 'KYUREM',
    'KELDEOORDINARY FORM': 'KELDEO'
    
}

# To add to the appropriate regional pokedexes
MISSING_POKEMON = {
    'Sinnoh': ['BURMY SANDY CLOAK', 'BURMY TRASH CLOAK', 'WORMADAM SANDY CLOAK', 'WORMADAM TRASH CLOAK', 'HEAT ROTOM', 'WASH ROTOM', 'MOW ROTOM', 'FROST ROTOM', 'FAN ROTOM',
               'GIRATINA DISTORTION FORME'],
    'Unova (Black/White)': ['DARMANITAN ZEN MODE', 'MELOETTA PIROUETTE FORME', 'TORNADUS THERIAN FORME', 'THUNDURUS THERIAN FORME', 'LANDORUS THERIAN FORME',
                            'WHITE KYUREM', 'BLACK KYUREM'],
    'Unova (Black2/White2)': ['CASTFORM SUNNY FORM', 'CASTFORM RAINY FORM', 'CASTFORM SNOWY FORM', 'DARMANITAN ZEN MODE', 'TORNADUS THERIAN FORME', 'THUNDURUS THERIAN FORME', 
                              'LANDORUS THERIAN FORME', 'WHITE KYUREM', 'BLACK KYUREM']
}


# MOVESETS AND NATURES
GEN3_MOVE_TYPES = ['bug','dark','dragon','electric','fighting','fire','flying','ghost','grass','ground','ice','normal','poison','psychict','rock','water','steel']
GEN4_MOVE_TYPES = ['bug','dark','dragon','electric','fighting','fire','flying','ghost','grass','ground','ice','normal','poison','psychict','rock','water','steel']
GEN5_MOVE_TYPES = ['bug','dark','dragon','electric','fighting','fire','flying','ghost','grass','ground','ice','normal','poison','psychict','rock','water','steel']
POKEMON_LIST = pokedex_df['Pokémon'].tolist()
POKEMON_NAME_CORRECTIONS = {
    # Change names to match urls
    "nidoran♀": "nidoran-f",
    "nidoran♂": "nidoran-m",
    "mr. mime": "mr-mime",
    "mime jr.": "mime-jr",
    "farfetch'd": "farfetchd",
    # Replace normal forms with standard names used in urls
    "deoxys normal forme": "deoxys",
    "burmy plant cloak": "burmy",
    "wormadam plant cloak": "wormadam",
    "shaymin land forme": "shaymin",
    "basculin red-striped form": "basculin",
    "darmanitan standard mode": "darmanitan",
    "tornadus incarnate forme": "tornadus",
    "thundurus incarnate forme": "thundurus",
    "landorus incarnate forme": "landorus",
    "meloetta aria forme": "meloetta",
    "keldeo resolute form": "keldeo"
}
REMOVE_LIST = ['CASTFORM SUNNY FORM','CASTFORM RAINY FORM','CASTFORM SNOWY FORM','DEOXYS ATTACK FORME','DEOXYS DEFENSE FORME',
             'DEOXYS SPEED FORME','BURMY SANDY CLOAK','BURMY TRASH CLOAK','WORMADAM SANDY CLOAK','WORMADAM TRASH CLOAK','ROTOM HEAT FORME',
             'ROTOM WASH FORME','ROTOM FROST FORME','ROTOM FAN FORME','ROTOM MOW FORME','GIRATINA DISTORTION FORME','SHAYMIN SKY FORME',
             'BASCULIN WHITE-STRIPED FORM','BASCULIN BLUE-STRIPED FORM','DARMANITAN ZEN MODE','TORNADUS THERIAN FORME',
             'THUNDURUS THERIAN FORME','LANDORUS THERIAN FORME','MELOETTA PIROUETTE FORME','KYUREM WHITE','KYUREM BLACK',
             'KELDEO RESOLUTE FORM']


# TYPES
TYPES = ['bug','dark','dragon','electric','fighting','fire','flying','ghost','grass','ground','ice','normal','poison','psychic','rock','water','steel']


# WIDGETS
# Key matching game to regional pokedex columns in national pokedex dataframe for subsequent filtering 
REGIONAL_POKEDEX_DICT = {
    'FireRed/LeafGreen': 'Kanto Pokedex',
    'Ruby/Sapphire/Emerald': 'Hoenn Pokedex',
    'Diamond/Pearl/Platinum': 'Sinnoh Pokedex',
    'HeartGold/SoulSilver': 'Johto Pokedex',
    'Black/White': 'Unova Pokedex (Black/White)',
    'Black2/White2': 'Unova Pokedex (Black2/White2)'
}
# Key matching game to corresponding pokemon moveset dictionaries
POKEMON_MOVESETS_DICT = {
    'FireRed/LeafGreen': gen3_movesets,
    'Ruby/Sapphire/Emerald': gen3_movesets,
    'Diamond/Pearl/Platinum': gen4_movesets,
    'HeartGold/SoulSilver': gen4_movesets,
    'Black/White': gen5_movesets,
    'Black2/White2': gen5_movesets
}
# Key matching game to corresponding move dictionaries
MOVE_TYPE_DICT = {
    'FireRed/LeafGreen': gen3_moves,
    'Ruby/Sapphire/Emerald': gen3_moves,
    'Diamond/Pearl/Platinum': gen4_moves,
    'HeartGold/SoulSilver': gen4_moves,
    'Black/White': gen5_moves,
    'Black2/White2': gen5_moves
}
# Key matching game to corresponding move dictionaries
ITEMS_DICT = {
    'Held Items': held_items,
    'General Items': general_items,
    'Berries': berries,
    'Battle Items': battle_items,
    'Medicine': medicine,
    'Pokeballs': pokeballs,
    # TMs unique to each game
    'FireRed/LeafGreen': FRLG_df,
    'Ruby/Sapphire/Emerald': RSE_df,
    'Diamond/Pearl/Platinum': DPP_df,
    'HeartGold/SoulSilver': HGSS_df,
    'Black/White': BW_df,
    'Black2/White2': BW2_df
}